<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Search by meaning

**[Search by meaning](https://learning.nextia-ai.com/courses/search/m05/search-by-meaning/)** · Search and Ranking: Find the Best Match · Module 5, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** search with embeddings: score every product with one matrix product, measure dense search with NDCG@10, and encode a new query.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 40 MB), and about 200 MB for the optional model in part 4. No account and no key. |
| **You should know** | Vectors and cosine similarity (the mathematics course), and [Detect what the query asks for](https://learning.nextia-ai.com/courses/search/m04/detect-what-the-query-asks-for/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0 and PyTorch 2.15.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m05/search-by-meaning/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M05-L01-search-by-meaning/search-by-meaning-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, sentence-transformers 6.2.0, torch 2.15.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 sentence-transformers==6.2.0 torch==2.15.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'sentence-transformers', 'torch']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 41 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/embeddings/products.npy": "f422870c4ab75364fad883fa8858bd7f31f2591bad07753a8f713a0872824ab0",
    "data/embeddings/queries.npy": "5333da6bd9ac44ec1bf92ddf916ec60cb557466b90b5075711f59c8e61f96253",
    "data/embeddings/model.json": "b5d1088cde17e0eb1bb9565f2e1e45c8cfb5b6a545ae72e44404d2429d6d9af4",
    "compare.py": "72695e7ef1aed02bb7da8927785c03657cadcea9f07ca63b368ea2a33f2d54b6",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Products and queries as vectors

A **bi-encoder** reads a text and writes an **embedding**: a list of numbers (here 384) where texts with similar meaning get similar numbers. The course team encoded every product once (title, class and the start of the description) with IBM's granite-embedding-97m-multilingual-r2 (Apache-2.0), and every query of the course, so this lesson does not wait. Run the next cell.

In [ ]:
cat = catalogue()
q = queries()
j = judgments()
P = np.load("data/embeddings/products.npy").astype(np.float32)   # one row per product
Q = np.load("data/embeddings/queries.npy").astype(np.float32)    # one row per query of queries.csv
meta = json.load(open("data/embeddings/model.json"))
print(P.shape, Q.shape, meta["model"])
print("length of a product vector:", round(float(np.linalg.norm(P[0])), 3))

> **Check.** 40066 product vectors of 384 numbers, each of length 1. For vectors of length 1, the dot product **is** the cosine similarity (the mathematics course explains why).

## 2. One query by meaning

Run the next cell: "rose gold lounge" against every product with one matrix product.

In [ ]:
row = q.index[q["query_id"] == "W-088"][0]
scores = P @ Q[row]                     # cosine similarity: the vectors have length 1
top = np.argsort(-scores)[:5]
print(q.loc[row, "query"])
show(cat["product_id"].to_numpy()[top], j["W-088"], n=5)

## 3. Your turn: dense search on the validation queries

Complete `dense_run`: for each query row, the 10 products with the highest score. Then compute the mean NDCG@10 on the English validation queries.

In [ ]:
def dense_run(rows):
    ids = cat["product_id"].to_numpy()
    out = {}
    for r in rows:
        ...   # the 10 products with the highest P @ Q[r], as a list of product IDs, in out[query_id]
    return out

val = q[(q["split"] == "validation") & (q["source"] == "wands")]
ndcg_dense = ...   # the mean NDCG@10 of dense_run(val.index)
print(ndcg_dense)

In [ ]:
expect('ndcg_dense', ndcg_dense, 0.64, tolerance=0.002)

> **Check.** About 0.64. In the course's measurements, tuned BM25 does better than dense search on these English queries: dense search finds products by meaning, but it misses exact words that matter. The next lesson finds where.

## 4. A query that the course never saw (optional, downloads about 200 MB)

The next cell downloads the model at its fixed version and encodes a new query, as a search service does for every search.

> **Predict.** "seat for the garden" shares no word with "patio bench". Will a bench come up?

In [ ]:
from sentence_transformers import SentenceTransformer
import torch
model = SentenceTransformer(meta["model"], revision=meta["revision"], device="cpu",
                            model_kwargs={"dtype": torch.float32})
model.max_seq_length = meta["max_tokens"]
v = model.encode(["seat for the garden"], normalize_embeddings=True)[0]
show(cat["product_id"].to_numpy()[np.argsort(-(P @ v))[:5]], n=5)

In the project: `search query "seat for the garden" --run dense` and `search eval --run dense --by qtype`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Where dense search fails](https://learning.nextia-ai.com/courses/search/m05/where-dense-search-fails/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m05/search-by-meaning/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.